# Last.fm batch recommendations

Run `recommend_album()` over a subset of `albums.csv` and write results to `datasets/lastfm_recommendations_<subset>_<strategy>.csv`.

Re-run the batch cell to resume — rows with `status` already set are skipped.

In [1]:
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from lastfm_albums import recommend_album, set_request_delay

In [2]:
DATA_DIR = Path("../datasets")
SOURCE_PATH = DATA_DIR / "albums.csv"

TRACK_SELECTION = "random"  # top_listener | random | top_n_tracks
TOP_N = 3           # number of seed tracks used by top_n_tracks strategy
N_RECS = 1          # recommendations to return per album
FETCH_FLOOR = 20    # similar tracks fetched per seed track (larger = more candidates, more API calls)
REQUEST_DELAY_SEC = .79   # seconds between API requests
RANDOM_SEED = 42    # seed for random track selection (strategy "random")
SAVE_EVERY = 10     # checkpoint to CSV every N albums

## Subset

Pick which albums to process. Output file name follows the subset key and `TRACK_SELECTION`.

In [3]:
SUBSETS = {
    "all": lambda df: df,
    "3plus": lambda df: df.loc[df["review_count"] > 2],
    "test": lambda df: df.sample(250),
    "1k": lambda df: df.sample(1000),
}

In [4]:
SUBSET = "1k" 
MAX_ALBUMS = None  # e.g. 10 for a dry run

In [5]:
albums = SUBSETS[SUBSET](pd.read_csv(SOURCE_PATH))
if MAX_ALBUMS is not None:
    albums = albums.head(MAX_ALBUMS)

strategy_key = (
    f"top_n_tracks_{TOP_N}" if TRACK_SELECTION == "top_n_tracks" else TRACK_SELECTION
)
OUTPUT_PATH = DATA_DIR / f"lastfm_recommendations_{SUBSET}_{strategy_key}.csv"
print(f"Subset: {SUBSET} ({len(albums):,} albums)")
print(f"Strategy: {TRACK_SELECTION}")
print(f"Output: {OUTPUT_PATH}")

Subset: 1k (1,000 albums)
Strategy: random
Output: ../datasets/lastfm_recommendations_1k_random.csv


In [6]:
RESULT_COLS = ["strategy", "status", "error", "seed_track"] + [
    f"rec_{rank}_{field}"
    for rank in range(1, N_RECS + 1)
    for field in ("album", "artist", "score")
]

if OUTPUT_PATH.exists():
    saved = pd.read_csv(OUTPUT_PATH)
    work = albums.merge(saved[["album_id"] + RESULT_COLS], on="album_id", how="left")
else:
    work = albums.assign(**{col: pd.NA for col in RESULT_COLS})

work["strategy"] = work["strategy"].astype("string")
work.loc[work["strategy"].isna() | (work["strategy"].str.strip() == ""), "strategy"] = TRACK_SELECTION
work["status"] = work["status"].astype("string")
work["error"] = work["error"].astype("string")
work["seed_track"] = work["seed_track"].astype("string")
for rank in range(1, N_RECS + 1):
    work[f"rec_{rank}_album"] = work[f"rec_{rank}_album"].astype("string")
    work[f"rec_{rank}_artist"] = work[f"rec_{rank}_artist"].astype("string")
    work[f"rec_{rank}_score"] = pd.to_numeric(work[f"rec_{rank}_score"], errors="coerce")

pending = work.index[work["status"].isna() | (work["status"].astype(str).str.strip() == "")]
print(f"Strategy: {TRACK_SELECTION}")
print(f"Pending: {len(pending):,} / {len(work):,}")
work.head(3)

Strategy: random
Pending: 1,000 / 1,000


,album_id,artist,album,review_count,strategy,status,error,seed_track,rec_1_album,rec_1_artist,rec_1_score
21296,21438,Saweetie,High Maintenance EP,1,random,<NA>,<NA>,<NA>,<NA>,<NA>,NaN
22017,22192,Sinking Ships,Out of Key Harmony,1,random,<NA>,<NA>,<NA>,<NA>,<NA>,NaN
6283,6203,Default Genders,Pain Mop Girl 2020,1,random,<NA>,<NA>,<NA>,<NA>,<NA>,NaN


In [ ]:
set_request_delay(REQUEST_DELAY_SEC)

since_save = 0
for idx in tqdm(pending, desc="Last.fm batch"):
    row = work.loc[idx]
    album_seed = RANDOM_SEED + int(row["album_id"])
    work.loc[idx, "strategy"] = TRACK_SELECTION

    try:
        _, seed_track, recs, _ = recommend_album(
            row["album"],
            artist=row["artist"],
            n_recs=N_RECS,
            fetch_floor=FETCH_FLOOR,
            track_selection=TRACK_SELECTION,
            top_n=TOP_N,
            random_seed=album_seed,
            clear_cache=False,
        )
        if recs.empty:
            work.loc[idx, ["status", "error"]] = ["no_results", pd.NA]
        else:
            work.loc[idx, "status"] = "ok"
            work.loc[idx, "error"] = pd.NA
            work.loc[idx, "seed_track"] = seed_track["name"] if seed_track else pd.NA
            for rank, (_, rec) in enumerate(recs.iterrows(), start=1):
                if rank > N_RECS:
                    break
                work.loc[idx, f"rec_{rank}_album"] = rec["album"]
                work.loc[idx, f"rec_{rank}_artist"] = rec["artist"]
                work.loc[idx, f"rec_{rank}_score"] = rec["similarity_score"]
    except Exception as exc:
        work.loc[idx, ["status", "error"]] = ["error", str(exc)[:500]]

    since_save += 1
    if since_save >= SAVE_EVERY:
        work.to_csv(OUTPUT_PATH, index=False)
        since_save = 0

work.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

Last.fm batch:   0%|          | 0/1000 [00:00<?, ?it/s]

In [8]:
print(work["status"].value_counts(dropna=False))
work.loc[work["status"] == "ok", ["artist", "album", "seed_track", "rec_1_album", "rec_1_artist"]].head()

status
<NA>          899
ok             54
no_results     30
error          17
Name: count, dtype: int64[pyarrow]


,artist,album,seed_track,rec_1_album,rec_1_artist
12844,Keane,Perfect Symmetry,You Haven't Told Me Anything,The Boy With No Name,Travis
15917,MellowHype,BLACKENEDWHITE,64,EARL,Earl Sweatshirt
9570,Gravenhurst,The Western Lands,Song Among The Pine,The Golden Archipelago,Shearwater
6997,Duke Special,Oh Pioneer,Little Black Fish,Songs for the Road,David Ford
5180,Courtney Pine,House of Legends,Song of the Maroons,Enter the Spirit,Bob Berg


In [ ]:
# # Reset failed rows, then re-run the batch cell
# retry = work.index[work["status"].isin(["error", "no_results"])]
# work.loc[retry, "status"] = pd.NA
# work.loc[retry, "error"] = pd.NA
# work.loc[retry, "seed_track"] = pd.NA
# for rank in range(1, N_RECS + 1):
#     work.loc[retry, f"rec_{rank}_album"] = pd.NA
#     work.loc[retry, f"rec_{rank}_artist"] = pd.NA
#     work.loc[retry, f"rec_{rank}_score"] = float("nan")

# pending = work.index[work["status"].isna() | (work["status"].astype(str).str.strip() == "")]
# print(f"Reset {len(retry):,} rows; pending: {len(pending):,} — re-run batch cell")

Reset 469 rows; pending: 469 — re-run batch cell
